In [ ]:
import sys
from pathlib import Path

# Add repo root to Python path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import pandas as pd
from pathlib import Path
from src.data import load_raw_csv
from src.prep import prepare_complaints_df, stratified_cap_per_class

TEXT_COL = "narrative"
LABEL_COL = "Product"
DATE_COL  = "Date received"

# Load and prepare the raw complaints data
df = load_raw_csv("../data/raw")
df = prepare_complaints_df(df, text_col=TEXT_COL, label_col=LABEL_COL, date_col=DATE_COL, min_chars=20)

# Keep only the top N most frequent labels
top_n = 10
top_labels = df[LABEL_COL].value_counts().head(top_n).index
df = df[df[LABEL_COL].isin(top_labels)].copy()

# Create a smaller stratified dataset with a cap on the number of samples per class
CAP_PER_CLASS = 10000
SEED = 42
df_small = stratified_cap_per_class(df, label_col=LABEL_COL, cap_per_class=CAP_PER_CLASS, random_state=SEED)

print(df_small.shape)
print(df_small[LABEL_COL].value_counts())

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Embed the complaint texts using a pre-trained SentenceTransformer model
model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

texts = df_small["text_clean"].tolist()

# Compute embeddings in batches
emb = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

emb.shape

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# Cluster the embeddings using MiniBatchKMeans
K = 30
kmeans = MiniBatchKMeans(
    n_clusters=K,
    batch_size=4096,
    random_state=SEED,
    n_init="auto"
)

cluster_id = kmeans.fit_predict(emb)
df_small["cluster_id"] = cluster_id

df_small["cluster_id"].value_counts().head()

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Identify top terms per cluster using TF-IDF
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=10
)
X_tfidf = vectorizer.fit_transform(df_small["text_clean"])
terms = np.array(vectorizer.get_feature_names_out())

# Function to get top terms for a given cluster
def top_terms_for_cluster(c_id: int, topn: int = 6):
    idx = np.where(df_small["cluster_id"].values == c_id)[0]
    if len(idx) == 0:
        return []
    # mean tfidf within cluster
    mean_tfidf = X_tfidf[idx].mean(axis=0)
    mean_tfidf = np.asarray(mean_tfidf).ravel()
    top_idx = mean_tfidf.argsort()[-topn:][::-1]
    return terms[top_idx].tolist()

cluster_keywords = {c: top_terms_for_cluster(c, topn=6) for c in range(K)}
cluster_label = {c: " | ".join(cluster_keywords[c][:4]) for c in range(K)}

df_small["cluster_label"] = df_small["cluster_id"].map(cluster_label)
df_small[["cluster_id","cluster_label"]].drop_duplicates().sort_values("cluster_id").head(10)

In [ ]:
# Save cluster labels and keywords to CSV
Path("../reports").mkdir(exist_ok=True)
pd.DataFrame([
    {"cluster_id": c, "label": cluster_label[c], "keywords": ", ".join(cluster_keywords[c])}
    for c in range(K)
]).to_csv("../reports/cluster_labels.csv", index=False)

In [ ]:
import numpy as np

# Get cluster centers and normalize for cosine similarity
centers = kmeans.cluster_centers_.astype("float32")
# embeddings were normalized; centers are not necessarily normalized
# normalize centers for cosine similarity
centers = centers / np.linalg.norm(centers, axis=1, keepdims=True)

def top_examples(c_id: int, n: int = 5):
    idx = np.where(df_small["cluster_id"].values == c_id)[0]
    if len(idx) == 0:
        return []
    sims = emb[idx] @ centers[c_id].T
    top_local = idx[np.argsort(-sims)[:n]]
    rows = df_small.iloc[top_local][[DATE_COL, LABEL_COL, "text_clean"]]
    return rows.to_dict(orient="records")

examples_by_cluster = {c: top_examples(c, n=5) for c in range(K)}

In [ ]:
# Compute weekly counts per cluster
df_small["week"] = df_small[DATE_COL].dt.to_period("W").dt.start_time

# Compute weekly counts per cluster
weekly = (
    df_small.groupby(["week", "cluster_id", "cluster_label"], as_index=False)
            .size()
            .rename(columns={"size": "n"})
)

weekly.head()

In [ ]:
import pandas as pd
import numpy as np

# Identify emerging clusters based on recent growth
all_weeks = sorted(weekly["week"].unique())
if len(all_weeks) < 8:
    print("Warning: not enough weeks for stable emerging logic; still generating report.")

last_week = all_weeks[-1]
recent_weeks = all_weeks[-2:]         # last 2 weeks
prev_weeks = all_weeks[-6:-2]         # 4 weeks before that

recent = weekly[weekly["week"].isin(recent_weeks)].groupby(["cluster_id","cluster_label"])["n"].mean()
prev   = weekly[weekly["week"].isin(prev_weeks)].groupby(["cluster_id","cluster_label"])["n"].mean()

# Combine previous and recent averages to compute growth rates
emerge = (
    pd.concat([prev.rename("prev_avg"), recent.rename("recent_avg")], axis=1)
      .fillna(0)
      .reset_index()
)

emerge["growth_rate"] = np.where(emerge["prev_avg"] > 0, emerge["recent_avg"] / emerge["prev_avg"], np.inf)

MIN_PREV_AVG = 10
GROWTH_THRESH = 1.5

# Identify emerging clusters based on thresholds
emerging = emerge[(emerge["prev_avg"] >= MIN_PREV_AVG) & (emerge["growth_rate"] >= GROWTH_THRESH)] \
            .sort_values(["growth_rate", "recent_avg"], ascending=False)

emerging.head(10)

In [ ]:
from pathlib import Path

# Generate markdown report
report_path = Path("../reports/theme_report.md")

top_clusters = df_small["cluster_id"].value_counts().head(10).index.tolist()

def md_escape(s: str) -> str:
    return str(s).replace("\n", " ").strip()

lines = []
lines.append("# Consumer Complaints Theme Report\n")
lines.append(f"- Dataset sample: top {top_n} Products, cap_per_class={CAP_PER_CLASS}, clusters(K)={K}\n")
lines.append(f"- Embeddings: {model_name}\n")
lines.append("\n## Top clusters by volume\n")
for c in top_clusters:
    lines.append(f"- **Cluster {c}**: `{cluster_label[c]}` (n={int((df_small['cluster_id']==c).sum())})\n")

lines.append("\n## Emerging themes (last 2 weeks vs previous 4 weeks)\n")
if len(emerging) == 0:
    lines.append("- No clusters met the emerging thresholds.\n")
else:
    for _, r in emerging.head(10).iterrows():
        lines.append(f"- **Cluster {int(r['cluster_id'])}** `{r['cluster_label']}`: prev_avg={r['prev_avg']:.1f}, recent_avg={r['recent_avg']:.1f}, growth={r['growth_rate']:.2f}x\n")

lines.append("\n## Cluster details (examples)\n")
for c in range(K):
    lines.append(f"\n### Cluster {c}: `{cluster_label[c]}`\n")
    lines.append(f"- Keywords: {', '.join(cluster_keywords[c])}\n")
    ex = examples_by_cluster[c]
    if not ex:
        lines.append("- No examples.\n")
        continue
    lines.append("- Top examples:\n")
    for e in ex:
        lines.append(f"  - ({str(e[DATE_COL])[:10]}) [{md_escape(e[LABEL_COL])}] {md_escape(e['text_clean'][:240])}...\n")

report_path.write_text("".join(lines), encoding="utf-8")
print("Wrote:", report_path)
